# Download and re-structure the ADARP dataset from Zenodo

In [ ]:
import urllib.request
import zipfile
import os
import numpy as np
import pandas as pd

files_path = "/content/sensor_data";

# download dataset
url = "https://zenodo.org/records/6640290/files/Sensor%20Data.zip?download=1"
zip_filepath = "/content/sensor_data.zip"
urllib.request.urlretrieve(url, zip_filepath)

# unzip
with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
    zip_ref.extractall(files_path)

# delete zip
os.remove(zip_filepath)

# download drink data
alcohol_use_data = pd.read_csv("https://raw.githubusercontent.com/AngeloKarugo/ADARP_Dataset/refs/heads/master/Participant%20TLFB.csv")

In [ ]:
!pip install heartpy
!pip install hrv-analysis

from datetime import datetime, timedelta
import heartpy as hp
import warnings
from hrvanalysis import get_time_domain_features, remove_outliers, remove_ectopic_beats, interpolate_nan_values

def calculate_rmssd_from_bvp_1min_windows(df: pd.DataFrame) -> np.ndarray:
    """
    Calculates RMSSD from BVP values over 1-minute windows.

    The RMSSD value for each timestamp will be the RMSSD computed from the BVP
    values within the 1-minute window that the timestamp falls into.
    For example, all timestamps between 00:00:00 and 00:00:59 (inclusive)
    will have the same RMSSD value, computed from BVP data in that minute.

    Args:
        df (pd.DataFrame): DataFrame with at least two columns:
                           'bvp_values' (raw BVP signal) and
                           'timestamp' (datetime objects).
                           Assumes bvp_values are sampled at 64 Hz.

    Returns:
        np.ndarray: A NumPy array of RMSSD values, with the same length as the
                    input DataFrame. Each element corresponds to the RMSSD for
                    the 1-minute window that its corresponding timestamp falls into.
                    Returns NaN for windows with insufficient data or if RMSSD cannot be computed.
    """
    if 'bvp_value' not in df.columns or 'timestamp' not in df.columns:
        raise ValueError("DataFrame must contain 'bvp_values' and 'timestamp' columns.")

    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        raise ValueError("The 'timestamp' column must be of datetime type.")

    sampling_rate = 64  # Hz, as specified
    window_size_seconds = 60 # 1-minute window

    # Ensure the DataFrame is sorted by timestamp for correct windowing
    df = df.sort_values(by='timestamp').copy()

    # Create a list to store the RMSSD values for each timestamp
    rmssd_values_list = []

    # Group by 5-minute intervals
    df_min_groups = df.groupby(pd.Grouper(key='timestamp', freq='5min'))

    for timestamp_group, group in df_min_groups:
        # Reset the index to a simple numerical one before processing with heartpy
        group_reset = group.reset_index(drop=True)

        group_bvp_vals = group_reset["bvp_value"]

        # Process the BVP data for the current 1-minute window
        try:
            wd, m = hp.process(hrdata=hp.filter_signal(group_bvp_vals, sample_rate=sampling_rate,
                                        cutoff=[0.8,3.5],
                                        order=3, filtertype='bandpass'), # Basic filtering for BVP
                               sample_rate=sampling_rate)

            # Get the RMSSD value for this window
            rmssd_value = m.get("rmssd", np.nan) # Use .get to handle potential missing key
        except Exception as e:
            # Handle potential errors during heartpy processing (e.g., insufficient data)
            print(f"Warning: Could not process BVP data for window starting at {timestamp_group}.")
            rmssd_value = np.nan

        # Append the RMSSD value for each row in the original group (before index reset)
        # This ensures the rmssd_values_list has the same length as the original df
        rmssd_values_list.extend([rmssd_value] * len(group))

    # Convert the list to a NumPy array
    return np.array(rmssd_values_list)

def calculate_rmssd_from_ibi(ibi_values: pd.Series, start_timestamp: datetime, window_size_seconds: int = 60) -> np.ndarray:
    """
    Calculates RMSSD from IBI values over 1-minute windows.

    The RMSSD value for each timestamp will be the RMSSD computed from the IBI
    values within the 1-minute window that the timestamp falls into.

    Args:
        ibi_values: values of time(ms) between heart beats
        start_timestamp: timestamp of the first IBI value
        window_size_seconds: size of the window in seconds

    Returns:
        np.ndarray: A NumPy array of RMSSD values, with the same length as the
                    input DataFrame. Each element corresponds to the RMSSD for
                    the 1-minute window that its corresponding timestamp falls into.
                    Returns NaN for windows with insufficient data or if RMSSD cannot be computed.
    """

    if not isinstance(ibi_values, pd.Series):
        raise TypeError("ibi_values must be a pandas Series.")
    if not isinstance(start_timestamp, datetime):
        raise TypeError("start_timestamp must be a datetime object.")
    if not isinstance(window_size_seconds, int) or window_size_seconds <= 0:
        raise ValueError("window_size_seconds must be a positive integer.")

    print(ibi_values[:100])

    # Convert IBI values from ms to seconds for easier time calculations
    ibi_seconds = ibi_values / 1000.0

    # Calculate cumulative times to get the timestamp for each IBI event
    # The first IBI occurs after 0 seconds from start_timestamp, subsequent IBIs add their value
    cumulative_times_seconds = ibi_seconds.cumsum()

    # Adjust cumulative_times_seconds so the first IBI is at start_timestamp + ibi_values[0] / 1000
    # and subsequent IBIs follow.
    # To get the *end* time of each IBI interval relative to start_timestamp
    # We add 0 to the start of the cumsum, then drop the last element
    # This means timestamp[i] is the time *at which* the i-th heartbeat occurred
    # relative to start_timestamp, making it easier to assign to windows.
    timestamps = [start_timestamp + timedelta(seconds=t) for t in cumulative_times_seconds]

    # Create a Series with timestamps as index
    ibi_series_with_timestamps = pd.Series(ibi_values.values, index=timestamps)

    # Initialize an array to store the RMSSD values, same length as input ibi_values
    rmssd_results = np.full(len(ibi_values), np.nan)

    # Determine the start and end of the entire data period
    data_start = ibi_series_with_timestamps.index.min()
    data_end = ibi_series_with_timestamps.index.max()

    # Iterate through windows
    current_window_start = data_start.floor(f'{window_size_seconds}s') # Align to nearest window start

    # Ensure current_window_start is not before data_start if data_start is very early in its "floor" window
    if current_window_start + timedelta(seconds=window_size_seconds) < data_start:
         current_window_start = data_start

    # If the first actual beat is before data_start, it should start from data_start's window

    # Iterate through windows
    while current_window_start <= data_end:
        current_window_end = current_window_start + timedelta(seconds=window_size_seconds)

        # Select IBI values within the current window
        # We need to select IBIs where the *end time* of the IBI falls within the window.
        # Since our timestamps are already the end times of the IBI, this is straightforward.
        window_ibi_series = ibi_series_with_timestamps.loc[
            (ibi_series_with_timestamps.index >= current_window_start) &
            (ibi_series_with_timestamps.index < current_window_end)
        ]

        if len(window_ibi_series) >= 2: # RMSSD requires at least 2 values to compute differences
            # Pass milliseconds to get_rmssd as it expects them
            rr_intervals = window_ibi_series.tolist()

            # remove outliers
            rr_intervals = remove_outliers(rr_intervals=rr_intervals,
                                                low_rri=300, high_rri=2000)

            # interpolate removed outliers
            rr_intervals = interpolate_nan_values(rr_intervals=rr_intervals,
                                                    interpolation_method='linear')

            # remove ectopic beats
            nn_intervals = remove_ectopic_beats(rr_intervals=rr_intervals, method="malik")

            # interpolate removed ectopic beats
            nn_intervals = interpolate_nan_values(rr_intervals=nn_intervals)

            rmssd_value = get_time_domain_features(nn_intervals).get("rmssd", np.nan)
        else:
            rmssd_value = np.nan

        # Assign the calculated RMSSD value to all original IBI indices
        # that fall within this window.
        # Find the original indices whose corresponding timestamps are within the current window.
        # This mapping is tricky if we don't have a direct index mapping.
        # Let's rebuild the timestamp array for easier lookup.
        original_timestamps_for_mapping = pd.Series(index=timestamps, data=range(len(ibi_values)))

        # Get the indices of the original ibi_values that fall into this window
        indices_in_window = original_timestamps_for_mapping.loc[
            (original_timestamps_for_mapping.index >= current_window_start) &
            (original_timestamps_for_mapping.index < current_window_end)
        ].values

        if len(indices_in_window) > 0:
            rmssd_results[indices_in_window] = rmssd_value

        # Move to the next window
        current_window_start = current_window_end

    return rmssd_results

In [ ]:
participant_names = participant_folder_names = [
    'Part 101C',
    'Part 102C',
    'Part 104C',
    'Part 105C',
    'Part 106C',
    'Part 107C',
    'Part 108C',
    'Part 109C',
    'Part 110C',
    'Part 111C',
    'Part 112C'
]

sensor_data_filenames = [
    "ACC.csv",
    "EDA.csv",
    "TEMP.csv",
    "IBI.csv"
]

sensor_modalities = [
    "acc",
    "eda",
    "temp",
    "ibi"
]

def read_participant_sensor_data(participant_folder_name: str)->pd.DataFrame:
    participant_folder_path = files_path + "/Sensor Data/" + participant_folder_name

    participant_subfolders = os.listdir(participant_folder_path)

    data = {
        "acc": [],  # Initialize as list to store DataFrames
        "eda": [],
        "temp": [],
        "ibi": [],
    }

    for folder in participant_subfolders:
        participant_subfolder_path = participant_folder_path + "/" + folder

        for index, sensor_modality in enumerate(sensor_modalities):
            data_path = participant_subfolder_path + "/" + sensor_data_filenames[index]

            with open(data_path, 'r') as f:
                if sensor_modality == "acc":
                    timestamp_str = f.readline().strip().split(",")[0]
                    frequency_str = f.readline().strip().split(",")[0]
                elif sensor_modality == "ibi":
                    timestamp_str = f.readline().strip().split(",")[0]
                else:
                    timestamp_str = f.readline().strip()
                    frequency_str = f.readline().strip()

                data_values = [line.strip() for line in f]

            if not timestamp_str:
                continue

            start_timestamp = pd.to_datetime(float(timestamp_str), unit="s")

            if sensor_modality != "ibi":
                # Convert data_values to numeric, handling potential errors
                frequency = float(frequency_str)

                time_deltas = pd.to_timedelta(range(len(data_values)), unit='s') / frequency

                timestamps = start_timestamp + time_deltas

            if sensor_modality == "acc":
                values_split = [row.split(",") for row in data_values]

                # Convert the split values to floats
                values_split = [[abs(float(val)) for val in row] for row in values_split]

                # Create the DataFrame
                df = pd.DataFrame(values_split, columns=['x', 'y', 'z'])
                df['magnitude'] = df['x'] + df['y'] + df['z']
                df['timestamp'] = timestamps
            elif sensor_modality == "bvp":
                # get HRV(RMSSD) from BVP sensor readings
                data_values = pd.to_numeric(data_values, errors='coerce')
                df = pd.DataFrame({'bvp_value': data_values, 'timestamp': timestamps})

                rmssd_values = calculate_rmssd_from_bvp_1min_windows(df)

                df['rmssd_value'] = rmssd_values
            elif sensor_modality == "ibi":
                values_split = [row.split(",") for row in data_values]

                # Convert the split values to floats
                values_split = np.array([[abs(float(val)) for val in row] for row in values_split])

                # Create the DataFrame
                df = pd.DataFrame()

                df["ibi_value"] = values_split[:,1]

                print(df["ibi_value"].shape)

                df["timestamp"] = start_timestamp + pd.to_timedelta(values_split[:,0], unit='s')

                # get HRV(RMSSD) from IBI values
                df["ibi_rmssd_value"] = calculate_rmssd_from_ibi(df["ibi_value"]*1000, start_timestamp, 60)
            else:
                # Convert data_values to numeric, handling potential errors
                data_values = pd.to_numeric(data_values, errors='coerce')
                df = pd.DataFrame({'value': data_values, 'timestamp': timestamps})

            df = df.set_index('timestamp')

            # Assign the result of resample back to df
            df = df.resample("min").mean()

            data[sensor_modality].append(df) # Append DataFrame to the list

    patientDf = pd.DataFrame()

    # Concatenate DataFrames after the loop
    for sensor_modality in sensor_modalities:
        if data[sensor_modality]:
            combined_df = pd.concat(data[sensor_modality])

            combined_df = combined_df.groupby(level=0).mean()

            data[sensor_modality] = combined_df.sort_index() # sort by timestamps

            if sensor_modality == "acc":
                patientDf[f'{sensor_modality}_magnitude'] = data[sensor_modality]["magnitude"]
            elif sensor_modality == "bvp":
                patientDf[f'{sensor_modality}_value'] = data[sensor_modality]["bvp_value"]
                patientDf['rmssd'] = data[sensor_modality]["rmssd_value"]
            elif sensor_modality == "ibi":
                # patientDf[f'{sensor_modality}_value'] = data[sensor_modality]["ibi_value"]
                patientDf['ibi_rmssd_value'] = data[sensor_modality]["ibi_rmssd_value"]
            else:
                patientDf[f'{sensor_modality}_value'] = data[sensor_modality]["value"]
        else:
            data[sensor_modality] = pd.DataFrame() # Ensure it's an empty DataFrame if no data

    patientDf["participantName"] = participant_folder_name

    return patientDf

allPatientsDf = pd.DataFrame()

for participant in participant_names:
    patientDf = read_participant_sensor_data(participant)
    allPatientsDf = pd.concat([allPatientsDf, patientDf])

allPatientsDf = allPatientsDf.reset_index()

allPatientsDf.to_csv("allPatientsSensorData.csv")